In [1]:
# # 必要なパッケージの読み込み
# library(caret)
# library(Metrics)
# library(ggplot2)
# library(lattice)

# # 対象ファイル名一覧
# file_names <- c(
#   "n_base.csv", "n_base_OH.csv", "n_base_FP.csv",
#   "n_all.csv", "n_all_OH.csv", "n_all_FP.csv",
#   "m_base.csv", "m_base_OH.csv", "m_base_FP.csv",
#   "m_all.csv", "m_all_OH.csv", "m_all_FP.csv"
# )

# # データパスと日付
# base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/"
# today <- format(Sys.Date(), "%Y%m%d")

# # 評価指標の初期化
# target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
# metric_names <- c("Best_k", "R2", "RMSE", "MAE", "RPD", "n_samples", "n_features")
# result_matrix <- matrix(nrow = length(metric_names) * length(target_vars), ncol = length(file_names))
# rownames(result_matrix) <- as.vector(t(outer(metric_names, target_vars, paste, sep = "_")))
# colnames(result_matrix) <- file_names

# # メイン処理ループ
# for (file in file_names) {
#   filepath <- paste0(base_path, file)
#   cat("\n=== Processing:", file, "===\n")
#   df_all <- read.csv(filepath)
#   cat("Final dataset size:", dim(df_all)[1], dim(df_all)[2], "\n")

#   feature_vars <- setdiff(colnames(df_all), target_vars)

#   for (target_var in target_vars) {
#     cat("\n---\nTraining model for:", target_var, "on", file, "\n")
#     df <- df_all[, c(feature_vars, target_var)]
#     df <- df[complete.cases(df), ]

#     # モデルチューニング設定
#     tune_grid <- expand.grid(k = seq(1, 20, by = 2))  # 奇数のみ
#     ctrl <- trainControl(method = "cv", number = 10)

#     model <- train(
#       formula(paste(target_var, "~ .")),
#       data = df,
#       method = "knn",
#       metric = "RMSE",
#       trControl = ctrl,
#       tuneGrid = tune_grid
#     )

#     pred <- predict(model, df)
#     obs <- df[[target_var]]

#     R2 <- round(cor(pred, obs)^2, 3)
#     RMSE_val <- round(rmse(obs, pred), 3)
#     MAE_val <- round(mae(obs, pred), 3)
#     RPD_val <- round(sd(obs) / RMSE_val, 3)

#     cat("Best k value:", model$bestTune$k, "\n")
#     cat(file, target_var, ": R2 =", R2, ", RMSE =", RMSE_val, ", MAE =", MAE_val, ", RPD =", RPD_val, "\n")

#     # 結果保存
#     result_matrix[paste0("Best_k_", target_var), file] <- model$bestTune$k
#     result_matrix[paste0("R2_", target_var), file] <- R2
#     result_matrix[paste0("RMSE_", target_var), file] <- RMSE_val
#     result_matrix[paste0("MAE_", target_var), file] <- MAE_val
#     result_matrix[paste0("RPD_", target_var), file] <- RPD_val
#     result_matrix[paste0("n_samples_", target_var), file] <- nrow(df)
#     result_matrix[paste0("n_features_", target_var), file] <- ncol(df) - 1

#     # プロット設定
#     get_axis_limit <- function(values) {
#       max_val <- max(values, na.rm = TRUE)
#       if (max_val <= 1.5) return(ceiling(max_val / 0.1) * 0.1)
#       if (max_val <= 5) return(ceiling(max_val / 0.5) * 0.5)
#       if (max_val <= 30) return(ceiling(max_val / 2) * 2)
#       return(ceiling(max_val / 5) * 5)
#     }
#     range_max <- get_axis_limit(c(obs, pred))

#     # プロット生成
#     p <- ggplot(data.frame(Predicted = pred, Observed = obs), aes(x = Observed, y = Predicted)) +
#       geom_point(color = "blue", alpha = 0.7) +
#       geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
#       scale_x_continuous(limits = c(0, range_max)) +
#       scale_y_continuous(limits = c(0, range_max)) +
#       coord_fixed(ratio = 1) +
#       labs(
#         title = paste0(target_var, " Prediction (", file, ")"),
#         x = "Observed", y = "Predicted"
#       ) +
#       theme_minimal() +
#       annotate("text", x = range_max * 0.05, y = range_max * 0.95, hjust = 0, vjust = 1, size = 4,
#                label = paste0("RMSE = ", RMSE_val, "\nMAE = ", MAE_val, "\nRPD = ", RPD_val)) +
#       annotate("text", x = range_max * 0.95, y = range_max * 0.05, hjust = 1, vjust = 0, size = 5,
#                label = paste0("R² = ", R2))

#     # モデル保存
#     model_data_bundle <- list(model = model, training_data = df)
#     rds_file <- paste0("knn_model_data_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".rds")
#     saveRDS(model_data_bundle, file = rds_file)

#     # プロット保存
#     plot_file <- paste0("knn_plot_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".pdf")
#     pdf(plot_file, width = 6, height = 6)
#     print(p)
#     dev.off()
#   }
# }

# # 結果の保存
# output_file <- paste0("knn_summary_", today, ".csv")
# write.csv(result_matrix, output_file, row.names = TRUE)
# cat("\nSummary saved as", output_file, "\n")


Loading required package: ggplot2

Loading required package: lattice


Attaching package: 'Metrics'


The following objects are masked from 'package:caret':

    precision, recall





=== Processing: n_base.csv ===
Final dataset size: 358 12 

---
Training model for: Jsc on n_base.csv 
Best k value: 1 
n_base.csv Jsc : R2 = 1 , RMSE = 0 , MAE = 0 , RPD = Inf 

---
Training model for: Voc on n_base.csv 
Best k value: 3 
n_base.csv Voc : R2 = 0.869 , RMSE = 0.055 , MAE = 0.034 , RPD = 2.757 

---
Training model for: FF on n_base.csv 
Best k value: 1 
n_base.csv FF : R2 = 1 , RMSE = 0 , MAE = 0 , RPD = Inf 

---
Training model for: PCEmax on n_base.csv 
Best k value: 1 
n_base.csv PCEmax : R2 = 1 , RMSE = 0 , MAE = 0 , RPD = Inf 

=== Processing: n_base_OH.csv ===
Final dataset size: 358 143 

---
Training model for: Jsc on n_base_OH.csv 
Best k value: 5 
n_base_OH.csv Jsc : R2 = 0.782 , RMSE = 2.369 , MAE = 1.708 , RPD = 2.136 

---
Training model for: Voc on n_base_OH.csv 
Best k value: 1 
n_base_OH.csv Voc : R2 = 1 , RMSE = 0 , MAE = 0 , RPD = Inf 

---
Training model for: FF on n_base_OH.csv 
Best k value: 3 
n_base_OH.csv FF : R2 = 0.706 , RMSE = 0.061 , MAE = 0.

In [2]:
#20250625_修正


In [3]:
# 必要なパッケージの読み込み
library(caret)
library(Metrics)
library(ggplot2)
library(lattice)

# 対象ファイル名一覧
file_names <- c(
  "n_base.csv", "n_base_OH.csv", "n_base_FP.csv",
  "n_all.csv", "n_all_OH.csv", "n_all_FP.csv",
  "m_base.csv", "m_base_OH.csv", "m_base_FP.csv",
  "m_all.csv", "m_all_OH.csv", "m_all_FP.csv"
)

# データパスと日付
base_path <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/20220127_PLS 202201 ALL/"
today <- format(Sys.Date(), "%Y%m%d")

# 評価指標の初期化
target_vars <- c("Jsc", "Voc", "FF", "PCEmax")
metric_names <- c("Best_k", "R2", "RMSE", "MAE", "RPD", "n_samples", "n_features")
result_matrix <- matrix(nrow = length(metric_names) * length(target_vars), ncol = length(file_names))
rownames(result_matrix) <- as.vector(t(outer(metric_names, target_vars, paste, sep = "_")))
colnames(result_matrix) <- file_names

# メイン処理ループ
for (file in file_names) {
  filepath <- paste0(base_path, file)
  cat("\n=== Processing:", file, "===\n")
  df_all <- read.csv(filepath)
  cat("Final dataset size:", dim(df_all)[1], dim(df_all)[2], "\n")

  feature_vars <- setdiff(colnames(df_all), target_vars)

  for (target_var in target_vars) {
    cat("\n---\nTraining model for:", target_var, "on", file, "\n")
    df <- df_all[, c(feature_vars, target_var)]
    df <- df[complete.cases(df), ]

    # 学習と交差検証の設定
    tune_grid <- expand.grid(k = seq(1, 20, by = 2))
    ctrl <- trainControl(method = "cv", number = 10, savePredictions = "final")

    model <- train(
      formula(paste(target_var, "~ .")),
      data = df,
      method = "knn",
      metric = "RMSE",
      trControl = ctrl,
      tuneGrid = tune_grid
    )

    # 交差検証結果から最良kのみ抽出して評価
    pred_df <- model$pred
    pred_df <- pred_df[pred_df$k == model$bestTune$k, ]
    pred <- pred_df$pred
    obs <- pred_df$obs

    R2 <- round(cor(pred, obs)^2, 3)
    RMSE_val <- round(rmse(obs, pred), 3)
    MAE_val <- round(mae(obs, pred), 3)
    RPD_val <- round(sd(obs) / RMSE_val, 3)

    cat("Best k value:", model$bestTune$k, "\n")
    cat(file, target_var, ": R2 =", R2, ", RMSE =", RMSE_val, ", MAE =", MAE_val, ", RPD =", RPD_val, "\n")

    # 結果保存
    result_matrix[paste0("Best_k_", target_var), file] <- model$bestTune$k
    result_matrix[paste0("R2_", target_var), file] <- R2
    result_matrix[paste0("RMSE_", target_var), file] <- RMSE_val
    result_matrix[paste0("MAE_", target_var), file] <- MAE_val
    result_matrix[paste0("RPD_", target_var), file] <- RPD_val
    result_matrix[paste0("n_samples_", target_var), file] <- nrow(df)
    result_matrix[paste0("n_features_", target_var), file] <- ncol(df) - 1

    # プロット設定関数
    get_axis_limit <- function(values) {
      max_val <- max(values, na.rm = TRUE)
      if (max_val <= 1.5) return(ceiling(max_val / 0.1) * 0.1)
      if (max_val <= 5) return(ceiling(max_val / 0.5) * 0.5)
      if (max_val <= 30) return(ceiling(max_val / 2) * 2)
      return(ceiling(max_val / 5) * 5)
    }
    range_max <- get_axis_limit(c(obs, pred))

    # プロット作成
    p <- ggplot(data.frame(Predicted = pred, Observed = obs), aes(x = Observed, y = Predicted)) +
      geom_point(color = "blue", alpha = 0.7) +
      geom_abline(slope = 1, intercept = 0, linetype = "dashed", color = "red") +
      scale_x_continuous(limits = c(0, range_max)) +
      scale_y_continuous(limits = c(0, range_max)) +
      coord_fixed(ratio = 1) +
      labs(
        title = paste0(target_var, " Prediction (", file, ")"),
        x = "Observed", y = "Predicted"
      ) +
      theme_minimal() +
      annotate("text", x = range_max * 0.05, y = range_max * 0.95, hjust = 0, vjust = 1, size = 4,
               label = paste0("RMSE = ", RMSE_val, "\nMAE = ", MAE_val, "\nRPD = ", RPD_val)) +
      annotate("text", x = range_max * 0.95, y = range_max * 0.05, hjust = 1, vjust = 0, size = 5,
               label = paste0("R² = ", R2))

    # モデル保存
    model_data_bundle <- list(model = model, training_data = df)
    rds_file <- paste0("20250625_knn_model_data_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".rds")
    saveRDS(model_data_bundle, file = rds_file)

    # プロット保存
    plot_file <- paste0("20250625_knn_plot_", tools::file_path_sans_ext(file), "_", target_var, "_", today, ".pdf")
    pdf(plot_file, width = 6, height = 6)
    print(p)
    dev.off()
  }
}

# 結果を保存
output_file <- paste0("20250625_knn_summary_", today, ".csv")
write.csv(result_matrix, output_file, row.names = TRUE)
cat("\n✅ Summary saved as", output_file, "\n")



=== Processing: n_base.csv ===
Final dataset size: 358 12 

---
Training model for: Jsc on n_base.csv 
Best k value: 3 
n_base.csv Jsc : R2 = 0.78 , RMSE = 2.381 , MAE = 1.711 , RPD = 2.125 

---
Training model for: Voc on n_base.csv 
Best k value: 3 
n_base.csv Voc : R2 = 0.717 , RMSE = 0.081 , MAE = 0.052 , RPD = 1.872 

---
Training model for: FF on n_base.csv 
Best k value: 1 
n_base.csv FF : R2 = 0.548 , RMSE = 0.079 , MAE = 0.058 , RPD = 1.411 

---
Training model for: PCEmax on n_base.csv 
Best k value: 1 
n_base.csv PCEmax : R2 = 0.771 , RMSE = 1.304 , MAE = 0.955 , RPD = 2.027 

=== Processing: n_base_OH.csv ===
Final dataset size: 358 143 

---
Training model for: Jsc on n_base_OH.csv 
Best k value: 7 
n_base_OH.csv Jsc : R2 = 0.677 , RMSE = 2.872 , MAE = 2.129 , RPD = 1.762 

---
Training model for: Voc on n_base_OH.csv 
Best k value: 1 
n_base_OH.csv Voc : R2 = 0.655 , RMSE = 0.095 , MAE = 0.051 , RPD = 1.596 

---
Training model for: FF on n_base_OH.csv 
Best k value: 3 
